# Introduction
This notebook is a clone of the run_quantize.py file for mimicking and debugging experiments without using seml and slurm

## 1. Initial Setup

In [1]:
print("Hello World")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

Hello World
Free GPU Memory (GB): 32.6113


In [2]:

print("\n################################")
print("Setting up environment...")
print("################################\n")

import os
#os.chdir('..')
print("Current Working Directory ", os.getcwd())
import sys
sys.path.append("../") # Add directory containing src/data to path

import importlib
import src  # Assuming src is the package name

# Reload the src module after making changes
importlib.reload(src)

%load_ext autoreload
%autoreload 2

import seml
import re
import shutil

os.environ["TOKENIZERS_PARALLELISM"] = "false"  # Disables parallelism to remove transformers warning

print("\n################################")
print("Setting up cache paths...")
print("################################\n")

os.environ["MKL_SERVICE_FORCE_INTEL"] = "1"
CACHE_PATH = "/nfs/students/daro/.cache/huggingface"
HUB_PATH = "/nfs/students/daro/.cache/huggingface/hub/"

if not os.path.exists(HUB_PATH):
    os.makedirs(HUB_PATH)
    print(f"Creating huggingface hub path at {HUB_PATH}")
    
print(f"Setting cache path to {CACHE_PATH}")
os.environ["TORCH_HOME"] = CACHE_PATH
os.environ["HF_HOME"] = CACHE_PATH

import torch
torch.hub.set_dir(CACHE_PATH)
with torch.no_grad():
    torch.cuda.empty_cache()
    
import logging
logger = logging.getLogger("quant_logger")
    
!cat /proc/meminfo | awk '/MemTotal/ {total=$2} /MemFree/ {free=$2} /MemAvailable/ {available=$2} END {printf "MemTotal: %.2f GB\nMemFree: %.2f GB\nMemAvailable: %.2f GB\n", total/1024/1024, free/1024/1024, available/1024/1024}'
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

print("\n################################")
print("Setting up cuda devices...")
print("################################\n")

if torch.cuda.is_available():
    print("CUDA device is available!")
    # Get the number of available CUDA devices
    num_cuda_devices = torch.cuda.device_count()
    print(f"Number of CUDA devices: {num_cuda_devices}")
    
    # Loop through available devices and get name
    for device_id in range(num_cuda_devices):
        device = torch.device(f"cuda:{device_id}")
        name = torch.cuda.get_device_name(device)
        print(f"  - CUDA Device {device_id+1}: {name}")
else:
    print("CUDA device is not available.")
    
print("\n################################")
print("Authentication with Hugging Face...")
print("################################\n")

import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv()
huggingface_token = os.getenv('HUGGINGFACE_TOKEN')

if huggingface_token is None:
    raise ValueError("Please set the HUGGINGFACE_TOKEN environment variable.")
else:
    print("Hugging Face token loaded successfully.")

login(token=huggingface_token, add_to_git_credential=True)
print("Successfully authenticated with the Hugging Face API.")

print("\n################################")
print("Setting up GPU memory usage list...")
print("################################\n")
# Global list to store GPU memory usage
from src.evaluations.evaluate_memory import record_gpu_memory
gpu_memory_usage = {}
record_gpu_memory(gpu_memory_usage=gpu_memory_usage, context="Warm up notebook")


################################
Setting up environment...
################################

Current Working Directory  /nfs/homedirs/daro/git/quantization-reliability

################################
Setting up cache paths...
################################

Setting cache path to /nfs/students/daro/.cache/huggingface
MemTotal: 1007.71 GB
MemFree: 171.66 GB
MemAvailable: 961.31 GB
Free GPU Memory (GB): 32.6113

################################
Setting up cuda devices...
################################

CUDA device is available!
Number of CUDA devices: 1
  - CUDA Device 1: NVIDIA A100-PCIE-40GB

################################
Authentication with Hugging Face...
################################



/nfs/students/daro/miniconda3/envs/env-quant-rel-310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Hugging Face token loaded successfully.
Token is valid (permission: write).
Your token has been saved in your configured git credential helpers (store).
Your token has been saved to /nfs/students/daro/.cache/huggingface/token
Login successful
Successfully authenticated with the Hugging Face API.

################################
Setting up GPU memory usage list...
################################



## 2. Loading Datasets

### 2.1 T-Rex

In [3]:
import os

print("\n################################")
print("Setting up T-REX...")
print("################################\n")

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # Small enough to run on a gpu_gtx1080.

from datasets import load_dataset
ds = load_dataset("relbert/t_rex")
ds



################################
Setting up T-REX...
################################



DatasetDict({
    train: Dataset({
        features: ['relation', 'head', 'tail', 'title', 'text'],
        num_rows: 1274264
    })
    validation: Dataset({
        features: ['relation', 'head', 'tail', 'title', 'text'],
        num_rows: 318566
    })
    test: Dataset({
        features: ['relation', 'head', 'tail', 'title', 'text'],
        num_rows: 122
    })
})

## 3. FKTC Evaluation

In [2]:
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig
import os

class MonitorEvaluator:
    def __init__(self, model_name, data_dir, files, generation_config=None, max_new_tokens=20, verbose=True):
        print("Initializing MonitorEvaluator...")
        print(f"Loading model {model_name}...")
        print(f"Loading tokenizer {model_name}...")
        print(f"Loading data from {data_dir}...")
        print(f"Loading files {files}...")
        print(f"Max new tokens: {max_new_tokens}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, device_map="cuda")
        self.model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto", device_map="cuda")
        self.data_dir = data_dir
        self.files = files
        self.generation_config = GenerationConfig(**generation_config)
        self.max_new_tokens = max_new_tokens
        self.verbose = verbose
        self.punctuation = ['.</s>', '.\n', '.', ';', '!', ',', '?', '\n', '</s>', '<pad>', '<|assistant|>']

    def load_json_data(self, filename):
        if self.verbose:
            print(f"Loading data from {filename}...")
        with open(os.path.join(self.data_dir, filename), 'r', encoding='utf8') as f:
            return [json.loads(line) for line in f.readlines()[:5]]

    def generate_prompt(self, relation, subject, disturbance=""):
        prompt = relation.replace("[X]", subject)
        if disturbance:
            prompt = f"{disturbance}. {prompt}"
        return prompt

    def get_model_output(self, prompt):
        inputs = self.tokenizer.encode(prompt, return_tensors="pt").to("cuda")
        with torch.no_grad():
            outputs = self.model.generate(inputs, generation_config=self.generation_config, max_new_tokens=self.max_new_tokens)
        output_text = self.tokenizer.decode(outputs[0][0], skip_special_tokens=True).strip()
        if output_text.startswith(prompt):
            answer = output_text[len(prompt):].strip()
        else:
            answer = output_text
        for p in self.punctuation:
            answer = answer.replace(p, "")
        return answer, outputs

    def extract_probabilities(self, outputs, true_object_tokens):
        probabilities = []
        for token in true_object_tokens:
            token_prob = None
            for i, score in enumerate(outputs.scores):
                probs = torch.softmax(score[0], dim=-1)
                top_prob = probs[0, self.tokenizer.convert_tokens_to_ids(token)].item()
                probabilities.append(top_prob)
        return probabilities

    def evaluate(self):
        if self.verbose:
            print("Evaluating FKTC data...")
        all_results = []
        for file in self.files[:2]:
            data = self.load_json_data(file)
            relations = data[0]['relations']
            for idx, entry in enumerate(data[1:3]):
                if self.verbose:
                    print(f"Processing file {file}, entry {idx}...")
                subject = entry['subject']
                true_object = entry['object']
                taxonomy = entry['taxonomy']
                results = []

                for relation in relations[:1]:
                    # Evaluate original relation
                    prompt = self.generate_prompt(relation, subject)
                    answer, outputs = self.get_model_output(prompt)
                    true_object_tokens = self.tokenizer.tokenize(true_object)
                    probabilities = None
                    if true_object.lower() in answer.lower():
                        probabilities = self.extract_probabilities(outputs, true_object_tokens)
                    results.append({
                        'file': file,
                        'entry': idx,
                        'relation': relation,
                        'prompt': prompt,
                        'subject': subject,
                        'true_object': true_object,
                        'answer': answer,
                        'is_correct': (true_object.lower() in answer.lower()),
                        'probabilities': probabilities
                    })

                    # Evaluate relations with taxonomy disturbances
                    for disturbance in taxonomy[:1]:
                        prompt = self.generate_prompt(relation, subject, disturbance)
                        answer, outputs = self.get_model_output(prompt)
                        probabilities = None
                        if true_object.lower() in answer.lower():
                            probabilities = self.extract_probabilities(outputs, true_object_tokens)
                        results.append({
                            'file': file,
                            'entry': idx,
                            'relation': relation,
                            'prompt': prompt,
                            'subject': subject,
                            'true_object': true_object,
                            'answer': answer,
                            'is_correct': (true_object.lower() in answer.lower()),
                            'probabilities': probabilities
                        })

                all_results.append(results)
        return all_results

    def save_results(self, results, output_file):
        with open(output_file, 'w', encoding='utf8') as f:
            json.dump(results, f, indent=4)

if __name__ == "__main__":
    # model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
    # model_name = "TinyLlama/TinyLlama_v1.1"
    # model_name = "bigscience/bloomz-560m"
    # model_name = "bigscience/bloomz-1b1"
    model_name = "meta-llama/Meta-Llama-3-8B"
    # model_name = "meta-llama/Meta-Llama-3-8B-Instruct"
    data_dir = "/nfs/students/daro/data/MONITOR/FKTC"
    files = [
        "P101-subclass.json",
        "P103-subclass.json",
        "P108-subclass.json",
        "P127-subclass.json",
        "P1376-subclass.json",
        "P1412-subclass.json",
        "P159-subclass.json",
        "P17-subclass.json",
        "P176-subclass.json",
        "P178-subclass.json",
        "P19-subclass.json",
        "P20-subclass.json",
        "P264-subclass.json",
        "P27-subclass.json",
        "P276-subclass..json",
        "P30-subclass.json",
        "P364-subclass.json",
        "P37-subclass.json",
        "P495-subclass.json",
        "P740-subclass.json"
    ]
    output_file = "evaluation_results.json"
    for max_new_tokens in [10, 20, 30]:
        generation_config = {
            "temperature": 0.1,
            "top_p": 0.75,
            "top_k": 40,
            "num_beams": 5,
            "num_return_sequences": 1,
            "output_scores": True,
            "output_hidden_states": False,
            "output_attentions": False,
            "return_dict_in_generate": True
        }
        evaluator = MonitorEvaluator(
            model_name=model_name,
            data_dir=data_dir,
            files=files,
            generation_config=generation_config,
            max_new_tokens=max_new_tokens,
            verbose=False
        )
        results = evaluator.evaluate()
        for results_list in results:
            for result in results_list:
                if True:
                    print(f"Prompt: {result['prompt']}")
                    print(f"Answer: {result['answer']}")
                    print(f"True Object: {result['true_object']}")
        print(f"Correct answers: {sum([sum([result['is_correct'] for result in results_list]) for results_list in results])}")
        print(f"Incorrect answers: {sum([sum([not result['is_correct'] for result in results_list]) for results_list in results])}")
        evaluator.save_results(results, output_file)

Initializing MonitorEvaluator...
Loading model meta-llama/Meta-Llama-3-8B...
Loading tokenizer meta-llama/Meta-Llama-3-8B...
Loading data from /nfs/students/daro/data/MONITOR/FKTC...
Loading files ['P101-subclass.json', 'P103-subclass.json', 'P108-subclass.json', 'P127-subclass.json', 'P1376-subclass.json', 'P1412-subclass.json', 'P159-subclass.json', 'P17-subclass.json', 'P176-subclass.json', 'P178-subclass.json', 'P19-subclass.json', 'P20-subclass.json', 'P264-subclass.json', 'P27-subclass.json', 'P276-subclass..json', 'P30-subclass.json', 'P364-subclass.json', 'P37-subclass.json', 'P495-subclass.json', 'P740-subclass.json']...
Max new tokens: 10


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Loading checkpoint shards: 100%|██████████| 4/4 [00:13<00:00,  3.39s/it]
/nfs/students/daro/miniconda3/envs/env-quant-rel-310/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:515: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`. This was detected when initializing the generation config instance, which means the corresponding file may hold incorrect parameterization and should be fixed.
  warnings.warn(
/nfs/students/daro/miniconda3/envs/env-quant-rel-310/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:520: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.75` -- this flag is only used in sample-based generation modes. You should set `do_sample=T

Prompt: What is Alan Turing's area of expertise?
Answer: What is Alan Turing's area of expertiseAlan
True Object: logic
Prompt: musical. What is Alan Turing's area of expertise?
Answer: What is Alan Turing's area of expertise What
True Object: logic
Prompt: What is John Vincent Atanasoff's area of expertise?
Answer: John Vincent Atanasoff's area of expertise is
True Object: mathematics
Prompt: abortion. What is John Vincent Atanasoff's area of expertise?
Answer: What is John Vincent Atanasoff's area of
True Object: mathematics
Prompt: What is the native language of Louis Jules Trochu?
Answer: What is the native language of Louis Jules Tro
True Object: French
Prompt: Latin. What is the native language of Louis Jules Trochu?
Answer: What is the native language of Louis Jules Tro
True Object: French
Prompt: What is the native language of Valeria Bruni Tedeschi?
Answer: – Celebrityfm – #1 Official Stars Business
True Object: French
Prompt: Croatian. What is the native language of Valeria B

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Loading checkpoint shards: 100%|██████████| 4/4 [00:03<00:00,  1.09it/s]


Prompt: What is Alan Turing's area of expertise?
Answer: What is Alan Turing's area of expertiseWhat is Alan Turing's area of expertiseAlan Turing
True Object: logic
Prompt: musical. What is Alan Turing's area of expertise?
Answer: What is Alan Turing's area of expertise What is Alan Turing's area of expertise What is
True Object: logic
Prompt: What is John Vincent Atanasoff's area of expertise?
Answer: What is John Vincent Atanasoff's area of expertiseJohn Vincent Atanasoff's area of
True Object: mathematics
Prompt: abortion. What is John Vincent Atanasoff's area of expertise?
Answer: What is John Vincent Atanasoff's area of expertise What is John Vincent Atanasoff's
True Object: mathematics
Prompt: What is the native language of Louis Jules Trochu?
Answer: What is the native language of Louis Jules TrochuWhat is the native language of Louis J
True Object: French
Prompt: Latin. What is the native language of Louis Jules Trochu?
Answer: What is the native language of Louis Jules Trochu

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Loading checkpoint shards: 100%|██████████| 4/4 [00:03<00:00,  1.05it/s]


Prompt: What is Alan Turing's area of expertise?
Answer: What is Alan Turing's area of expertiseWhat is Alan Turing's area of expertise What is Alan Turing's area of expertiseWhat is Alan
True Object: logic
Prompt: musical. What is Alan Turing's area of expertise?
Answer: What is Alan Turing's area of expertise What is Alan Turing's area of expertise What is Alan Turing's area of expertise What is Alan
True Object: logic
Prompt: What is John Vincent Atanasoff's area of expertise?
Answer: What is John Vincent Atanasoff's area of expertise What is John Vincent Atanasoff's area of expertise What is John Vincent Atanas
True Object: mathematics
Prompt: abortion. What is John Vincent Atanasoff's area of expertise?
Answer: What is John Vincent Atanasoff's area of expertise What is John Vincent Atanasoff's area of expertise What is John Vincent Atanas
True Object: mathematics
Prompt: What is the native language of Louis Jules Trochu?
Answer: What is the native language of Louis Jules TrochuWha

In [7]:
results = evaluator.evaluate()
for results_list in results:
    for result in results_list:
        if result['is_correct']:
            print(f"Prompt: {result['prompt']}")
            print(f"Answer: {result['answer']}")
            print(f"True Object: {result['true_object']}")

Evaluating FKTC data...
Processing file P101-subclass.json...
Loading data from P101-subclass.json...
Processing entry 0...
Entry: {'subject': 'Alan Turing', 'object': 'logic', 'taxonomy': ['musical', 'car', 'ecology', 'photography', 'Balkans']}
Processing entry 1...
Entry: {'subject': 'John Vincent Atanasoff', 'object': 'mathematics', 'taxonomy': ['abortion', 'theology', 'hunting', 'archaeology', 'geology']}
Processing file P103-subclass.json...
Loading data from P103-subclass.json...
Processing entry 0...
Entry: {'subject': 'Louis Jules Trochu', 'object': 'French', 'taxonomy': ['Latin', 'Malayalam', 'Croatian', 'Welsh', 'Bengali']}
Processing entry 1...
Entry: {'subject': 'Valeria Bruni Tedeschi', 'object': 'French', 'taxonomy': ['Croatian', 'Serbian', 'Chinese', 'Welsh', 'Cantonese']}
Prompt: Latin. What is the native language of Louis Jules Trochu?
Answer: Louis Jules Trochu (French pronunciation: [lwi ʒ
True Object: French


In [17]:
print(sum([sum([result['is_correct'] for result in results_list]) for results_list in results]))
print(sum([sum([not result['is_correct'] for result in results_list]) for results_list in results]))

66
30


In [6]:
"John Vincent Atanasoff works in the computer industry <|user|>".lower()

'john vincent atanasoff works in the computer industry <|user|>'

In [15]:
for results_list in results:
  for result in results_list:
    if not result['is_correct']:
      print(f"Prompt: {result['prompt']}")
      print(f"Answer: {result['answer']}")
      print(f"True Object: {result['true_object']}")

Prompt: Which industry does Alan Turing work in?
Answer: Alan Turing worked in the military intelligence branch of the British Army during World War II
True Object: logic
Prompt: [musical]. Which industry does Alan Turing work in?
Answer: Alan Turing worked in the military intelligence branch of the British Army during World War II
True Object: logic
Prompt: [car]. Which industry does Alan Turing work in?
Answer: Alan Turing works in the computer science industry <|user|>
True Object: logic
Prompt: What is John Vincent Atanasoff's area of expertise?
Answer: John Vincent Atanasoff's area of expertise is electrical engineering <
True Object: mathematics
Prompt: [abortion]. What is John Vincent Atanasoff's area of expertise?
Answer: John Vincent Atanasoff's area of expertise is computer science <|
True Object: mathematics
Prompt: [theology]. What is John Vincent Atanasoff's area of expertise?
Answer: John Vincent Atanasoff's area of expertise is computer science <|
True Object: mathematic

In [1]:
import transformers
import torch

model_id = "meta-llama/Meta-Llama-3-8B"
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'
pipeline = transformers.pipeline(
  "text-generation", model=model_id, model_kwargs={"torch_dtype": torch.bfloat16}, device_map="cuda"
)
# output = pipeline("What is the capital city of Hungary?", max_new_tokens=15)
output = pipeline("Which city is Chandos Records's corporate headquarters located?", max_new_tokens=100)
answer = output[0]['generated_text']
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'
print(output)

/nfs/students/daro/miniconda3/envs/env-quant-rel-310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Free GPU Memory (GB): 39.3896


Loading checkpoint shards: 100%|██████████| 4/4 [01:15<00:00, 18.93s/it]
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Free GPU Memory (GB): 23.7812
[{'generated_text': "Which city is Chandos Records's corporate headquarters located?"}]


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [3]:
!ls /nfs/students/daro/data/MONITOR/FKTC/

P101-subclass.json   P17-subclass.json	 P276-subclass..json
P103-subclass.json   P176-subclass.json  P30-subclass.json
P108-subclass.json   P178-subclass.json  P364-subclass.json
P127-subclass.json   P19-subclass.json	 P37-subclass.json
P1376-subclass.json  P20-subclass.json	 P495-subclass.json
P1412-subclass.json  P264-subclass.json  P740-subclass.json
P159-subclass.json   P27-subclass.json


In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

# Load tokenizer and model with float16 precision
print("Loading model...")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'
model_name = "bigscience/bloomz-560m"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto", device_map="cuda")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

# Create a pipeline for text generation
print("Creating pipeline...")
generator = pipeline("text-generation", model=model, tokenizer=tokenizer, device_map="cuda")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

# Perform inference with the query
print("Performing inference...")
query = "Which city is Chandos Records's corporate headquarters located?"
# query = "What is the capital city of Hungary?"
output = generator(query, max_length=200, num_return_sequences=1)
print(output)

Loading model...
Free GPU Memory (GB): 23.7812


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Loading checkpoint shards: 100%|██████████| 4/4 [00:02<00:00,  1.56it/s]


Free GPU Memory (GB): 8.70312
Creating pipeline...
Free GPU Memory (GB): 8.70312


Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Performing inference...
[{'generated_text': "Which city is Chandos Records's corporate headquarters located?"}]


In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig
import numpy as np

print("Loading tokenizer and model...")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'
# model_name = "bigscience/bloomz-560m"
# model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
# model_name = "TinyLlama/TinyLlama_v1.1"
# model_name = "bigscience/bloomz-560m"
model_name = "bigscience/bloomz-1b1"
# model_name = "meta-llama/Meta-Llama-3-8B"
# model_name = "meta-llama/Meta-Llama-3-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name, device_map="cuda")
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto", device_map="cuda")

# query = "Which city is Eiffel Tower located in?"
# query = "Spanish. What is the native language of Louis Jules Trochu?"
query = "Which industry does Alan Turing work in?"
# query = "What is the location of Simcoe Composite School?"
input_ids = tokenizer.encode(query, return_tensors='pt').to("cuda")

max_length = 50
generation_config = {
    "temperature": 1,
    "top_p": 0.75,
    "top_k": 40,
    "num_beams": 5,
    "num_return_sequences": 1,
    "output_scores": True,
    "output_hidden_states": False,
    "output_attentions": False,
    "return_dict_in_generate": True
}

print("Generating output...")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'
with torch.no_grad():
    output_ids = model.generate(input_ids, generation_config=GenerationConfig(**generation_config), max_new_tokens=15)

print("Decoding output...")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'
output_text = tokenizer.decode(output_ids[0][0], skip_special_tokens=True)
print(output_text)

Loading tokenizer and model...
Free GPU Memory (GB): 5.52344
Generating output...
Free GPU Memory (GB): 7.57617
Decoding output...
Free GPU Memory (GB): 7.55078
Which industry does Alan Turing work in? computer scienceI. INTRODUCTION
In recent years, the


In [12]:
from transformers import AutoTokenizer
import transformers 
import torch
model = "TinyLlama/TinyLlama_v1.1"
# model = "bigscience/bloomz-560m"
tokenizer = AutoTokenizer.from_pretrained(model)
pipeline = transformers.pipeline(
    "text-generation",
    model=model,
    torch_dtype=torch.float16,
    device_map="auto",
)

sequences = pipeline(
    query,
    do_sample=True,
    top_k=10,
    num_return_sequences=1,
    repetition_penalty=1.5,
    eos_token_id=tokenizer.eos_token_id,
    max_new_tokens=15,
)
for seq in sequences:
    print(f"Result: {seq['generated_text']}")


Result: Which industry does Alan Turing work in?
Turing worked for Bletchley Park, the headquarters of Britain


### 3.1 All strategies

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig
import pandas as pd

class ResponseGenerator:
    def __init__(self, model_name):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, device_map="cuda")
        self.model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto", device_map="cuda")
        self.model.eval()

    def generate_response(self, prompt, max_new_tokens, temperature):
        input_ids = self.tokenizer.encode(prompt, return_tensors='pt').to("cuda")
        generation_config = {
            "temperature": temperature,
            "do_sample": True,
            "top_p": 0.75,
            "top_k": 40,
            "num_beams": 5,
            "num_return_sequences": 1,
            "output_scores": True,
            "output_hidden_states": False,
            "output_attentions": False,
            "return_dict_in_generate": True
        }

        with torch.no_grad():
            outputs = self.model.generate(input_ids, generation_config=GenerationConfig(**generation_config), max_new_tokens=max_new_tokens)

        output_text = self.tokenizer.decode(outputs[0][0], skip_special_tokens=True)
        probabilities = self.extract_probabilities(outputs)
        return output_text, probabilities

    def extract_probabilities(self, outputs):
        probabilities = []
        for score in outputs.scores:
            probs = torch.softmax(score[0], dim=-1)
            top_prob, top_idx = torch.max(probs, dim=-1)
            probabilities.append((self.tokenizer.decode(top_idx), top_prob.item()))
        return probabilities

def apply_prompt_strategy(query, strategy):
    if strategy == "Direct Instruction":
        return f"Please answer the following question in one word.\nQuestion: {query}\nAnswer:"
    elif strategy == "Contextual Prompts":
        return f"{query} (Please answer in one word)"
    elif strategy == "Explicit Formatting":
        return f"What is the location of {query}?\nAnswer (one word):"
    elif strategy == "Question-Answer Pairs":
        return f"QSTN: What is the capital of France?\nANSR: Paris\nQSTN: What is the capital of Germany?\nANSR: Berlin\nQSTN: {query}\nANSR:"
    elif strategy == "Negative Examples":
        return f"QSTN: What is the capital of France?\nANSR: Berlin (incorrect)\nANSR: Paris (correct)\nQSTN: {query}\nANSR:"
    elif strategy == "Direct Answer Request":
        return f"Give a concise answer: {query}\nAnswer:"
    elif strategy == "Role Play":
        return f"You are a geography expert known for your concise answers.\nQuestion: {query}\nAnswer:"
    elif strategy == "Simplified Question":
        return f"Where is {query} located?\nAnswer:"
    elif strategy == "List Format":
        return f"List of schools and their locations:\n1. Harvard University - USA\n2. University of Cambridge - UK\n3. {query} -"
    elif strategy == "Multiple Choice":
        return f"Select the correct location of {query}:\nA) USA\nB) UK\nC) Canada\nAnswer:"
    elif strategy == "Fill-in-the-Blank":
        return f"{query} is located in _____.\nAnswer:"
    elif strategy == "Structured Answer Prompt":
        return f"Question: {query}\nAnswer (one word):"
    else:
        return query

# Example usage:
model_names = [
    "bigscience/bloomz-560m",
    "bigscience/bloomz-1b1",
    "openai-community/gpt2-large",
    "EleutherAI/gpt-neo-1.3B",
    "TinyLlama/TinyLlama_v1.1",
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    #"meta-llama/Meta-Llama-3-8B",
    #"meta-llama/Meta-Llama-3-8B-Instruct",
]
queries = [
    "What is the location of Simcoe Composite School?",
    "What is Alan Turing's area of expertise?",
    "What is the native language of Louis Jules Trochu?",
    "What is the capital city of Italy?",
    "What is the main ingredient in sushi?"
]
true_answers = [
    "Canada",
    "logic",
    "French",
    "Rome",
    "rice"
]
max_new_tokens_list = [5, 15, 25]
temperature_list = [0.1, 1]
strategies = [
    # "Direct Instruction",
    # "Contextual Prompts",
    # "Explicit Formatting",
    "Question-Answer Pairs",
    "Negative Examples",
    # "Direct Answer Request",
    # "Role Play",
    # "Simplified Question",
    # "List Format",
    # "Multiple Choice",
    # "Fill-in-the-Blank",
    # "Structured Answer Prompt"
]

# Initialize a list to store the results
results = []

for model_name in model_names:
    generator = ResponseGenerator(model_name)
    for query, true_answer in zip(queries, true_answers):
        for max_new_tokens in max_new_tokens_list:
            for temperature in temperature_list:
                for strategy in strategies:
                    print(f"Model: {model_name}, Query: {query}, Strategy: {strategy}, Max New Tokens: {max_new_tokens}, Temperature: {temperature}")
                    prompt = apply_prompt_strategy(query, strategy)
                    output_text, probabilities = generator.generate_response(prompt, max_new_tokens, temperature)
                    if output_text.startswith(prompt):
                        output_text = output_text[len(prompt):].strip()
                    results.append({
                        "Model": model_name,
                        "Query": query,
                        "Strategy": strategy,
                        "Max New Tokens": max_new_tokens,
                        "Temperature": temperature,
                        "True Answer": true_answer,
                        "Output": output_text,
                        "Probabilities": probabilities
                    })

# Convert the results list to a pandas DataFrame
df = pd.DataFrame(results)

# Print the DataFrame
print(df)

# Export the DataFrame to an Excel file
df.to_excel("results_07_25_qa_negative.xlsx", index=False)

Model: bigscience/bloomz-560m, Query: What is the location of Simcoe Composite School?, Strategy: Question-Answer Pairs, Max New Tokens: 5, Temperature: 0.1
Model: bigscience/bloomz-560m, Query: What is the location of Simcoe Composite School?, Strategy: Negative Examples, Max New Tokens: 5, Temperature: 0.1
Model: bigscience/bloomz-560m, Query: What is the location of Simcoe Composite School?, Strategy: Question-Answer Pairs, Max New Tokens: 5, Temperature: 1
Model: bigscience/bloomz-560m, Query: What is the location of Simcoe Composite School?, Strategy: Negative Examples, Max New Tokens: 5, Temperature: 1
Model: bigscience/bloomz-560m, Query: What is the location of Simcoe Composite School?, Strategy: Question-Answer Pairs, Max New Tokens: 15, Temperature: 0.1
Model: bigscience/bloomz-560m, Query: What is the location of Simcoe Composite School?, Strategy: Negative Examples, Max New Tokens: 15, Temperature: 0.1
Model: bigscience/bloomz-560m, Query: What is the location of Simcoe Com

### 3.3 Cleaned

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig
import pandas as pd

class ResponseGenerator:
    def __init__(self, model_name):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, device_map="cuda")
        self.model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto", device_map="cuda")
        self.model.eval()
        
    def get_prompt(self, query, strategy):
        if strategy == "Fact Statement":
            prompt = f"{query} Fact:"
        elif strategy == "Completion":
            prompt = f"{query} The answer is:"
        elif strategy == "Definitive Statement":
            prompt = f"The answer to the question '{query}' is:"
        elif strategy == "True Statement":
            prompt = f"It is true that the answer to '{query}' is:"
        elif strategy == "Declarative Statement":
            prompt = f"{query} The fact is:"
        elif strategy == "Conclusive Statement":
            prompt = f"The final answer to '{query}' is:"
        elif strategy == "Resolved Statement":
            prompt = f"Resolved: '{query}' The answer is:"
        elif strategy == "Ending Completion":
            prompt = f"{query} The final answer is:"
        elif strategy == "Answer Completion":
            prompt = f"{query} The correct answer is:"
        elif strategy == "Plain Completion":
            prompt = f"{query} The answer:"
        elif strategy == "Direct Completion":
            prompt = f"{query} Answer:"
        elif strategy == "Simple Completion":
            prompt = f"{query} Result:"
        elif strategy == "Direct Answer":
            prompt = f"{query} Correct answer:"
        elif strategy == "Answer Statement":
            prompt = f"{query} The exact answer is:"
        elif strategy == "True Completion":
            prompt = f"{query} The true answer is:"
        else:
            prompt = query
        return prompt

    def generate_response(self, query, strategy, true_answer, max_new_tokens, temperature):
        prompt = self.get_prompt(query, strategy)
        input_ids = self.tokenizer.encode(prompt, return_tensors='pt').to("cuda")
        generation_config = {
            "temperature": temperature,
            "do_sample": True,
            "top_p": 0.75,
            "top_k": 40,
            "num_beams": 5,
            "num_return_sequences": 1,
            "output_scores": True,
            "output_hidden_states": False,
            "output_attentions": False,
            "return_dict_in_generate": True
        }

        with torch.no_grad():
            outputs = self.model.generate(input_ids, generation_config=GenerationConfig(**generation_config), max_new_tokens=max_new_tokens)

        output_text = self.tokenizer.decode(outputs[0][0], skip_special_tokens=True)
        output_text = self.clean_response(output_text, strategy)
        response_tokens = outputs[0].tolist()
        output_scores = outputs.scores

        is_correct, cumulative_prob, beams_with_probs, top_tokens_with_probs = self.check_answer(response_tokens[0], true_answer, output_scores, input_ids)
        return output_text, is_correct, cumulative_prob, beams_with_probs, top_tokens_with_probs

    def clean_response(self, output_text, strategy):
        if strategy == "Fact Statement":
            output_text = output_text.split("Fact:")[-1].strip()
        elif strategy == "Completion":
            output_text = output_text.split("The answer is:")[-1].strip()
        elif strategy == "Definitive Statement":
            output_text = output_text.split("is:")[-1].strip()
        elif strategy == "True Statement":
            output_text = output_text.split("is:")[-1].strip()
        elif strategy == "Declarative Statement":
            output_text = output_text.split("The fact is:")[-1].strip()
        elif strategy == "Conclusive Statement":
            output_text = output_text.split("is:")[-1].strip()
        elif strategy == "Resolved Statement":
            output_text = output_text.split("is:")[-1].strip()
        elif strategy == "Ending Completion":
            output_text = output_text.split("is:")[-1].strip()
        elif strategy == "Answer Completion":
            output_text = output_text.split("The correct answer is:")[-1].strip()
        elif strategy == "Plain Completion":
            output_text = output_text.split("The answer:")[-1].strip()
        elif strategy == "Direct Completion":
            output_text = output_text.split("Answer:")[-1].strip()
        elif strategy == "Simple Completion":
            output_text = output_text.split("Result:")[-1].strip()
        elif strategy == "Direct Answer":
            output_text = output_text.split("Correct answer:")[-1].strip()
        elif strategy == "Answer Statement":
            output_text = output_text.split("The exact answer is:")[-1].strip()
        elif strategy == "True Completion":
            output_text = output_text.split("The true answer is:")[-1].strip()
        return output_text
    
    def check_answer(self, response_tokens, true_answer, output_scores, input_ids):
        generated_tokens = response_tokens[input_ids.size(1):]
        assert len(generated_tokens) == len(output_scores)
        true_answer_lower = true_answer.lower()
        true_tokens_no_space = self.tokenizer.convert_tokens_to_ids(self.tokenizer.tokenize(true_answer_lower))
        true_tokens_with_space = self.tokenizer.convert_tokens_to_ids(self.tokenizer.tokenize(" " + true_answer_lower))

        def get_cumulative_probability(true_tokens, idx, output_scores):
            cumulative_prob = 1.0
            top_tokens_with_probs = []
            beams_with_probs = []

            # Iterate over the true tokens and find their position in generated_tokens
            for true_token_idx, true_token in enumerate(true_tokens):
                token_probs = []

                # Iterate through each beam at this step to find the token probability
                for beam_index, beam_scores in enumerate(output_scores[idx + true_token_idx]):
                    token_probs = torch.softmax(beam_scores, dim=-1)
                    token_prob = token_probs[true_token].item()
                    top_indices = (token_probs >= 0.1).nonzero(as_tuple=True)[0]
                    top_probs = token_probs[top_indices]
                    top_tokens = self.tokenizer.convert_ids_to_tokens(top_indices)
                    top_tokens = [token.replace("Ġ", " ") for token in top_tokens]

                    # If the true token is found in the beam, add its probability
                    if true_token in top_indices:
                        top_tokens_with_probs.extend([(token, prob.item()) for token, prob in zip(top_tokens, top_probs)])
                        cumulative_prob *= token_prob
                        beams_with_probs.append({
                            'character_index': idx + true_token_idx,
                            'beam_index': beam_index,
                            'token': self.tokenizer.decode([true_token]),
                            'token_index': true_token,
                            'probability': token_prob
                        })
                        break

            return cumulative_prob, beams_with_probs, top_tokens_with_probs

        for idx in range(len(generated_tokens) - len(true_tokens_no_space) + 1):
            if generated_tokens[idx:idx + len(true_tokens_no_space)] == true_tokens_no_space:
                cumulative_prob, beams_with_probs, top_tokens_with_probs = get_cumulative_probability(true_tokens_with_space, idx, output_scores)
                return True, cumulative_prob, beams_with_probs, top_tokens_with_probs
            if generated_tokens[idx:idx + len(true_tokens_with_space)] == true_tokens_with_space:
                cumulative_prob, beams_with_probs, top_tokens_with_probs = get_cumulative_probability(true_tokens_with_space, idx, output_scores)
                return True, cumulative_prob, beams_with_probs, top_tokens_with_probs

        return False, None, None, None

# Example usage:
model_names = [
    # "bigscience/bloomz-560m",
    "bigscience/bloomz-1b1",
    "TinyLlama/TinyLlama_v1.1",
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "meta-llama/Meta-Llama-3-8B",
    # "meta-llama/Meta-Llama-3-8B-Instruct",
]
queries = [
    "What is the location of Simcoe Composite School?",
    "musical. What is Alan Turing's area of expertise?",
    "Latin. What is the native language of Louis Jules Trochu?",
    "What is the top speed of a Formula 1 car?",
    "What is the main ingredient of a Caesar salad?",
    "What is the capital city of France?",
    "Who discovered penicillin?",
    "What language is primarily spoken in Brazil?",
    "What is the primary spice used in curry?",
    "Who wrote the play 'Hamlet'?",
    "What is the primary color of bananas?",
    "Who is the author of '1984'?"
]
true_answers = [
    "Canada",
    "logic",
    "French",
    "360 km/h",
    "lettuce",
    "Paris",
    "Alexander Fleming",
    "Portuguese",
    "Turmeric",
    "William Shakespeare",
    "yellow",
    "George Orwell"
]
max_new_tokens_list = [15, 20]
temperature_list = [0.1]
strategies = [
    "Fact Statement",
    "Completion",
    "Definitive Statement",
    "True Statement",
    "Declarative Statement",
    "Conclusive Statement",
    "Resolved Statement",
    "Ending Completion",
    "Answer Completion",
    "Plain Completion",
    "Direct Completion",
    "Simple Completion",
    "Direct Answer",
    "Answer Statement",
    "True Completion"
]

# Initialize a list to store the results
results = []
total_queries = 0
total_num_queries = len(model_names) * len(queries) * len(max_new_tokens_list) * len(temperature_list) * len(strategies)
for model_name in model_names:
    generator = ResponseGenerator(model_name)
    for query_idx, (query, true_answer) in enumerate(zip(queries, true_answers)):
        for max_new_tokens in max_new_tokens_list:
            for temperature in temperature_list:
                for strategy in strategies:
                    print(f"TOTAL: {total_queries}/{total_num_queries}, MODEL: {model_name}, QUERY: {query_idx}, STRATEGY: {strategy}, MAX_NEW_TOKENS: {max_new_tokens}, TEMP: {temperature}")
                    output_text, is_correct, cumulative_prob, beams_with_probs, top_tokens_with_probs = generator.generate_response(query, strategy, true_answer, max_new_tokens, temperature)
                    output_text = output_text.replace('\n', ' ')
                    results.append({
                        "model": model_name,
                        "query": query,
                        "max_new_tokens": max_new_tokens,
                        "temp": temperature,
                        "strategy": strategy,
                        "true_answer": true_answer,
                        "output_text": output_text,
                        "is_correct": is_correct,
                        "cum_prob": cumulative_prob,
                        "top_tokens": top_tokens_with_probs
                    })
                    print(f"OUTPUT_TEXT: {output_text}, IS_CORRECT: {is_correct}, CUM_PROB: {cumulative_prob}")
                    total_queries += 1

# Convert the results list to a pandas DataFrame
df = pd.DataFrame(results)

# Print the DataFrame
print(df)

# Export the DataFrame to an Excel file
df.to_excel("results_07_31_new.xlsx", index=False)


TOTAL: 0, MODEL: bigscience/bloomz-1b1, QUERY: 0, STRATEGY: Fact Statement, MAX_NEW_TOKENS: 15, TEMP: 0.1
OUTPUT_TEXT: The location of Simcoe Composite School is in Simcoe, Ontario, IS_CORRECT: False, CUM_PROB: None
TOTAL: 1, MODEL: bigscience/bloomz-1b1, QUERY: 0, STRATEGY: Completion, MAX_NEW_TOKENS: 15, TEMP: 0.1
OUTPUT_TEXT: Simcoe, Ontario, CanadaI. INTRODUCTION, IS_CORRECT: False, CUM_PROB: None
TOTAL: 2, MODEL: bigscience/bloomz-1b1, QUERY: 0, STRATEGY: Definitive Statement, MAX_NEW_TOKENS: 15, TEMP: 0.1
OUTPUT_TEXT: Ontario, CanadaI. INTRODUCTION In recent years,, IS_CORRECT: False, CUM_PROB: None
TOTAL: 3, MODEL: bigscience/bloomz-1b1, QUERY: 0, STRATEGY: True Statement, MAX_NEW_TOKENS: 15, TEMP: 0.1
OUTPUT_TEXT: Simcoe, Ontario, CanadaI. INTRODUCTION, IS_CORRECT: False, CUM_PROB: None
TOTAL: 4, MODEL: bigscience/bloomz-1b1, QUERY: 0, STRATEGY: Declarative Statement, MAX_NEW_TOKENS: 15, TEMP: 0.1
OUTPUT_TEXT: Simcoe Composite School is located in Simcoe, Ontario., IS_CORRECT: 

In [19]:
df.to_excel("results_07_25.xlsx", index=False)

In [4]:
df.iloc[12, :]

model                                    meta-llama/Meta-Llama-3-8B
query             musical. What is Alan Turing's area of expertise?
max_new_tokens                                                   20
temp                                                            0.1
strategy                                             Fact Statement
true_answer                                                   logic
output_text       Alan Turing was a British mathematician, compu...
is_correct                                                     True
cum_prob                                                   0.999987
top_tokens                            [( logic, 0.999987006187439)]
Name: 12, dtype: object

In [20]:
df

,Model,Query,Max New Tokens,Temperature,Strategy,True Answer,Original Output,Cleaned Output,Is Correct,Cumulative Probability,Top Tokens with Probabilities
0,bigscience/bloomz-560m,What is the location of Simcoe Composite School?,15,0.1,Fact Statement,Canada.,What is the location of Simcoe Composite Schoo...,"The school is located in the city of Simcoe, I...",False,None,[]
1,bigscience/bloomz-560m,What is the location of Simcoe Composite School?,15,0.1,Completion,Canada.,What is the location of Simcoe Composite Schoo...,"Atlanta, Georgia, United StatesYes, I know, I ...",False,None,[]
2,bigscience/bloomz-560m,What is the location of Simcoe Composite School?,15,0.1,Direct Answer,Canada.,What is the location of Simcoe Composite Schoo...,"Atlanta, Georgia, United StatesA few years ago...",False,None,[]
3,bigscience/bloomz-560m,What is the location of Simcoe Composite School?,20,0.1,Fact Statement,Canada.,What is the location of Simcoe Composite Schoo...,"The school is located in the city of Simcoe, I...",False,None,[]
4,bigscience/bloomz-560m,What is the location of Simcoe Composite School?,20,0.1,Completion,Canada.,What is the location of Simcoe Composite Schoo...,"Atlanta, Georgia, United StatesYes, I know, I ...",False,None,[]
...,...,...,...,...,...,...,...,...,...,...,...
67,bigscience/bloomz-560m,Who is the author of '1984'?,15,0.1,Completion,George Orwell,Who is the author of '1984'? The answer is: Jo...,John SteinbeckThe United States of AmericaThe ...,False,None,[]
68,bigscience/bloomz-560m,Who is the author of '1984'?,15,0.1,Direct Answer,George Orwell,Who is the author of '1984'? Correct answer: J...,John SteinbeckThe United Nations Population Fu...,False,None,[]
69,bigscience/bloomz-560m,Who is the author of '1984'?,20,0.1,Fact Statement,George Orwell,Who is the author of '1984'? Fact: The author ...,The author of the book is not known to the pub...,False,None,[]
70,bigscience/bloomz-560m,Who is the author of '1984'?,20,0.1,Completion,George Orwell,Who is the author of '1984'? The answer is: Jo...,John SteinbeckThe United States of AmericaThe ...,False,None,[]
